**1. Import Libraries and Mount to Drive**

In [ ]:
# enables 4-bit/8-bit quantization and 32-bit optimizers (makes it possble for fine-tuning with less vram)
!pip install -U bitsandbytes

# import libraries
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model
from google.colab import drive
import os
import pandas as pd
from torch.nn.utils.rnn import pad_sequence
import math
import shutil
import time
from transformers import TrainerCallback
from google.colab import files

# mount to personal google drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 50.9 MB/s eta 0:00:00
Mounted at /content/drive


**2. Confirm GPU is Active**

In [ ]:
# finds out if the gpu is available
if not torch.cuda.is_available():
  print("GPU currently not available. Please select the correct runtime type (Runtime → Change runtime type → GPU).")
else:
  print(f"   - GPU detected: {torch.cuda.get_device_name(0)}") # displays the gpu name
  print(f"   - VRAM available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB") # displays how much memory the gpu has
  print(f"   - PyTorch CUDA version: {torch.version.cuda}") # displays what version of CUDA PyTorch is being used to talk to the GPU

  # tells pytorch to use the GPU
  device = torch.device("cuda")

   - GPU detected: NVIDIA RTX PRO 6000 Blackwell Server Edition
   - VRAM available: 102.0 GB
   - PyTorch CUDA version: 12.8


**3. Load Model + Tokenizer**

In [ ]:
# name of the pre-trained model being load
model_name = "meta-llama/Llama-3.2-3B-Instruct"

# load tokenizer for the model (coverts text into numbers / converts numbers back to text)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # llama has no pad token by default so borrow the eos token instead

# load the model in bfloat16 on the GPU
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="cuda",
)

# disable cache to save memory during training
model.config.use_cache = False

# align the model's pad token ID with the tokenizer so loss calculations ignore padded space
model.config.pad_token_id = tokenizer.pad_token_id

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

**4. Load Dataset (Med-MathInstruct)**

In [ ]:
# folder path to the medical + math dataset files
folder_path = "/content/drive/MyDrive/Fine-Tuning-Datasets/Med-MathInstruct"

# load the train and val sets for the medical dataset
train_df = pd.read_json(f"{folder_path}/combined_medical_math_train.jsonl", lines=True)
val_df   = pd.read_json(f"{folder_path}/combined_medical_math_val.jsonl",   lines=True)

# convert dataset to Hugging face format
train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)

# calculates full length of dataset
medical_ds = len(train_ds) + len(val_ds)

print(f"Medical + Math Dataset Size: {medical_ds}")
print(f"Train Dataset Size: {len(train_ds)}")
print(f"Val Dataset Size: {len(val_ds)}")

Medical + Math Dataset Size: 64784
Train Dataset Size: 61375
Val Dataset Size: 3409


**5. Convert Instruction Format to Llama-3.2-3B-Instruct Chat Templete**

In [ ]:
# formats each dataset row into the Llama 3 chat template
def format_instruction(sample):

    # system prompt left empty because it was adding date infromation the model did not need to learn
    system_prompt = ""

    # get the main question instruction and any extra details input from the data
    user_prompt = sample["instruction"]
    input_text = sample.get("input", "")

    # combines instruction and input together if exists to make one question
    if input_text and input_text.strip():
        user_prompt = f"{user_prompt}\n\n{input_text.strip()}"

    # manually format to Llama 3 chat template  https://www.llama.com/docs/model-cards-and-prompt-formats/llama3_1/
    text = (
      f"<|start_header_id|>system<|end_header_id|>\n\n"
      f"{system_prompt}<|eot_id|>"
      f"<|start_header_id|>user<|end_header_id|>\n\n"
      f"{user_prompt}<|eot_id|>"
      f"<|start_header_id|>assistant<|end_header_id|>\n\n"
      f"{sample['output']}<|eot_id|>"
    )

    return {"text": text} # returns the formated text

# applies the templates to the dataset
train_ds = train_ds.map(format_instruction)
val_ds = val_ds.map(format_instruction)

# confirms that the chat template was applied
print(train_ds[2]["text"])

Map:   0%|          | 0/61375 [00:00<?, ? examples/s]

Map:   0%|          | 0/3409 [00:00<?, ? examples/s]

<|start_header_id|>system<|end_header_id|>

<|eot_id|><|start_header_id|>user<|end_header_id|>

Solve the following clinical calculation. Extract the necessary variables from the patient note, show the step-by-step reasoning and formula used, and provide the final numeric result.

Patient Note: A 59-year-old man presents to general medical clinic for his yearly checkup. He has no complaints except for a dry cough. He has a past medical history of type II diabetes, hypertension, hyperlipidemia, asthma, and depression. His home medications are sitagliptin/metformin, lisinopril, atorvastatin, albuterol inhaler, and citalopram. His vitals signs are stable, with blood pressure 126/79 mmHg. Hemoglobin A1C is 6.3%, and creatinine is 1.3 g/dL. The remainder of his physical exam is unremarkable.

Question: What is patient's mean arterial pressure in mm Hg?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

The mean average pressure is computed by the formula 2/3 * (diastolic blood pressure

**6. Apply Tokenization**

In [ ]:
# tokenizes the data into numbers
def tokenize(sample):
  tokens = tokenizer(
      sample["text"],
      truncation=True,  # added as a safety net even though this was already caught in preprocessing
      max_length=2048,   # limits input length to 2048 tokens

      padding=False # handeled in custom data collector section 8
  )
  return tokens

# applies the tokenizer to each set
train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize,   batched=True)

# confirms tokenizer has been applied
print("Successfully applied tokenize")
print("\n")
print(train_ds[4]["input_ids"])
print("\n")
print(tokenizer.convert_ids_to_tokens(train_ds[2]["input_ids"]))

Map:   0%|          | 0/61375 [00:00<?, ? examples/s]

Map:   0%|          | 0/3409 [00:00<?, ? examples/s]

Successfully applied tokenize


[128000, 128006, 9125, 128007, 271, 128009, 128006, 882, 128007, 271, 50, 71306, 279, 2768, 5224, 482, 330, 86245, 389, 41906, 68642, 1288, 8891, 369, 17508, 74, 22317, 689, 1210, 128009, 128006, 78191, 128007, 271, 791, 5224, 330, 86245, 389, 41906, 68642, 1288, 8891, 369, 17508, 74, 22317, 689, 1, 3445, 430, 7931, 889, 527, 4737, 41906, 68642, 1288, 387, 46878, 323, 15870, 1817, 872, 62275, 5990, 304, 279, 6680, 4245, 311, 264, 4754, 3185, 2515, 2663, 17508, 74, 22317, 689, 11, 902, 374, 459, 671, 86336, 1579, 2237, 315, 62275, 304, 279, 6680, 382, 1271, 40821, 279, 5224, 11, 584, 649, 4148, 904, 26225, 4339, 323, 1304, 433, 810, 64694, 1473, 1, 5693, 70785, 6978, 1288, 8891, 369, 17508, 74, 22317, 689, 1210, 128009]


['<|begin_of_text|>', '<|start_header_id|>', 'system', '<|end_header_id|>', 'ĊĊ', '<|eot_id|>', '<|start_header_id|>', 'user', '<|end_header_id|>', 'ĊĊ', 'S', 'olve', 'Ġthe', 'Ġfollowing', 'Ġclinical', 'Ġcalculation', '.', 'ĠExtract', 'Ġ

**7. Prompt Masking - (Learn from assistant reply tokens only)**

In [ ]:
# sets the assistant header
assistant_header = "<|start_header_id|>assistant"

# mask non-assistant tokens so loss is only calculated on the assistant reply (output)
def mask_labels(sample):

  # grab the tokens and make a copy to edit
  input_ids = sample["input_ids"]
  labels = input_ids.copy()

  # converts assistant header text to token ids
  assistant_token_ids = tokenizer.encode(assistant_header, add_special_tokens=False)

  # stores how long the assistant header is
  header_len = len(assistant_token_ids)
  mask_end = 0

  # find where the assistant response starts in the input
  for i in range(len(input_ids) - header_len):
     if input_ids[i : i + header_len] == assistant_token_ids:
            mask_end = i + header_len + 2
            break

  # hide everything before the assistant reply so the model only learns from it
  labels[:mask_end] = [-100] * mask_end
  return {"labels": labels}

# apply masking to all rows so model only learns from assistant replies
train_ds = train_ds.map(mask_labels, batched=False, remove_columns=["text"])
val_ds   = val_ds.map(mask_labels,   batched=False, remove_columns=["text"])

# confirms prompt masking has been applied
print("Successfully applied prompt masking")
print("\n")
print("Input IDs:", train_ds[4]["input_ids"])
print("\n")
print("Labels:", train_ds[4]["labels"])

Map:   0%|          | 0/61375 [00:00<?, ? examples/s]

Map:   0%|          | 0/3409 [00:00<?, ? examples/s]

Successfully applied prompt masking


Input IDs: [128000, 128006, 9125, 128007, 271, 128009, 128006, 882, 128007, 271, 50, 71306, 279, 2768, 5224, 482, 330, 86245, 389, 41906, 68642, 1288, 8891, 369, 17508, 74, 22317, 689, 1210, 128009, 128006, 78191, 128007, 271, 791, 5224, 330, 86245, 389, 41906, 68642, 1288, 8891, 369, 17508, 74, 22317, 689, 1, 3445, 430, 7931, 889, 527, 4737, 41906, 68642, 1288, 387, 46878, 323, 15870, 1817, 872, 62275, 5990, 304, 279, 6680, 4245, 311, 264, 4754, 3185, 2515, 2663, 17508, 74, 22317, 689, 11, 902, 374, 459, 671, 86336, 1579, 2237, 315, 62275, 304, 279, 6680, 382, 1271, 40821, 279, 5224, 11, 584, 649, 4148, 904, 26225, 4339, 323, 1304, 433, 810, 64694, 1473, 1, 5693, 70785, 6978, 1288, 8891, 369, 17508, 74, 22317, 689, 1210, 128009]


Labels: [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 791, 52

**8. Masked Label Collactor**

In [ ]:
class MaskedLabelCollator:
  def __init__(self, tokenizer):
    self.tokenizer = tokenizer # used for pad token id later

  # pad all sequences to the longest one in the batch
  def __call__(self, features):

    # convert lists to tensors so the gpu can use them
    input_ids  = [torch.tensor(f["input_ids"])     for f in features]
    labels     = [torch.tensor(f["labels"])         for f in features]
    attn_masks = [torch.tensor(f["attention_mask"]) for f in features]

    # pad everything to same length
    input_ids  = pad_sequence(input_ids,  batch_first=True, padding_value=self.tokenizer.pad_token_id)
    labels     = pad_sequence(labels,     batch_first=True, padding_value=-100)  # -100 = ignored in loss
    attn_masks = pad_sequence(attn_masks, batch_first=True, padding_value=0)

    # return padded sequences to the trainer
    return {
      "input_ids":      input_ids,
      "labels":         labels,
      "attention_mask": attn_masks,
     }

 # set up the collator to pad batches
data_collator = MaskedLabelCollator(tokenizer)

# confirms mask label collator has been applied
print("Successfully applied mask label collator")
print("\n")
print("Input IDs:", train_ds[2]["input_ids"])
print("\n")
print("Labels:", train_ds[2]["labels"])
print("\n")
print("Attention mask:", train_ds[2]["attention_mask"])

Successfully applied mask label collator


Input IDs: [128000, 128006, 9125, 128007, 271, 128009, 128006, 882, 128007, 271, 50, 4035, 279, 2768, 14830, 22702, 13, 23673, 279, 5995, 7482, 505, 279, 8893, 5296, 11, 1501, 279, 3094, 14656, 30308, 33811, 323, 15150, 1511, 11, 323, 3493, 279, 1620, 25031, 1121, 382, 37692, 7181, 25, 362, 220, 2946, 4771, 6418, 893, 18911, 311, 4689, 6593, 28913, 369, 813, 45370, 1817, 455, 13, 1283, 706, 912, 21859, 3734, 369, 264, 9235, 40700, 13, 1283, 706, 264, 3347, 6593, 3925, 315, 955, 8105, 20335, 11, 63308, 11, 17508, 34215, 307, 22689, 11, 51643, 11, 323, 18710, 13, 5414, 2162, 31010, 527, 2503, 80156, 418, 258, 91328, 630, 258, 11, 41380, 258, 454, 31660, 11, 520, 269, 85, 561, 15111, 11, 453, 8248, 261, 337, 77773, 261, 11, 323, 272, 2223, 454, 2453, 13, 5414, 13458, 1147, 12195, 527, 15528, 11, 449, 6680, 7410, 220, 9390, 14, 4643, 9653, 39, 70, 13, 33924, 94855, 362, 16, 34, 374, 220, 21, 13, 18, 13689, 323, 6184, 83334, 374, 220, 16, 13, 18, 3

**9. Apply LoRA**

In [ ]:
lora_config = LoraConfig(
    r=32, # controls how many parameters are added during fine-tuning
    lora_alpha=64, # scaling factor unsloth rx2
    lora_dropout=0.1, # randomly turns off 10% of the neurons (helps generalization)
    bias="none", # dont train bias paramaters only LoRA weights
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"] # apply to all modules for a fair comparsion
)

# wraps the base model with LoRA adapter layers
model = get_peft_model(model, lora_config)

# confrims how many parameters are being trained vs frozen
model.print_trainable_parameters()

trainable params: 48,627,712 || all params: 3,261,377,536 || trainable%: 1.4910


**10. Training Arguments**

In [ ]:
training_args = TrainingArguments(
    output_dir="./medical-math_llama3_lora_output", # folder were model checkpoints are saved

    # training batch size 8 x 4 = 32
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,

    per_device_eval_batch_size=4, # eval batch size 4

    gradient_checkpointing=True, # saves memory but makes training slower

    optim="paged_adamw_8bit", # optimizor to use

    learning_rate=1e-4, # learning rate of the model
    lr_scheduler_type="cosine", # slowly reduce learning rate over training

    warmup_steps=200, # slowly warms up learning rate

    weight_decay=0.05, # reduce overfitting by shrinking large weights
    max_grad_norm=1.0,  # clip gradients to stop training going unstable

    num_train_epochs=3, # times model see training dataset

    # log every 20 steps, evaluate every 200 steps
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=200,

    # save checkpoint every 200 steps, keep only the 2 best, load best at end
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,

    # use bfloat16
    bf16=True,
    fp16=False,

    # speed up data loading
    dataloader_num_workers=4,
    dataloader_pin_memory=True,

    remove_unused_columns=False, # keep all columns including custom ones
    seed=42, # ensures results will be the same if run again
    data_seed=42, # ensures results are replicable
    report_to="none" # disable logging
)

**11. Trainer**

In [ ]:
# set up the trainer with model, data and training config
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
)

**12. Tranning**

In [ ]:
# callback to track best checkpoint time
class BestCheckpointCallback(TrainerCallback):
    def __init__(self):
        self.best_checkpoint_time = None
        self.last_best_checkpoint = None

    def on_evaluate(self, args, state, control, **kwargs):
        # if best checkpoint changes record time
        if state.best_model_checkpoint != self.last_best_checkpoint:
            self.last_best_checkpoint = state.best_model_checkpoint
            self.best_checkpoint_time = time.ctime()

# creates the callback
callback = BestCheckpointCallback()
trainer.add_callback(callback)

# store start time
train_start_time = time.time()
print(f"\nStarting LoRA fine-tuning at {time.ctime(train_start_time)}\n")

# train
trainer.train()

# store end time
train_end_time = time.time()

# print summary (clean output)
print("\nTraining completed")
print(f"   Finished at:           {time.ctime(train_end_time)}")
print(f"   Total steps:           {trainer.state.global_step:,}")
print(f"   Best checkpoint:       {trainer.state.best_model_checkpoint}")
print(f"   Best checkpoint time:  {callback.best_checkpoint_time}")
print(f"   Total training time:   {(train_end_time - train_start_time)/60:.2f} minutes")


Starting LoRA fine-tuning at Fri Apr 17 04:35:58 2026



Step,Training Loss,Validation Loss
200,0.647466,0.657578
400,0.588385,0.623015
600,0.592623,0.610407
800,0.540470,0.602780
1000,0.551282,0.596720
1200,0.570781,0.592253
1400,0.540480,0.589101
1600,0.556560,0.584822
1800,0.572834,0.581871
2000,0.505759,0.584558


Step,Training Loss,Validation Loss
200,0.647466,0.657578
400,0.588385,0.623015
600,0.592623,0.610407
800,0.540470,0.602780
1000,0.551282,0.596720
1200,0.570781,0.592253
1400,0.540480,0.589101
1600,0.556560,0.584822
1800,0.572834,0.581871
2000,0.505759,0.584558



Training completed
   Finished at:           Fri Apr 17 12:06:35 2026
   Total steps:           5,754
   Best checkpoint:       ./medical-math_llama3_lora_output/checkpoint-3800
   Best checkpoint time:  Fri Apr 17 09:50:24 2026
   Total training time:   450.61 minutes


**14. Save LoRA Adapter to Google Drive**

In [ ]:
# saves only the lightweight LoRA adapter weights not the full model
model.save_pretrained("medical-math_llama3_lora_adapter")
tokenizer.save_pretrained("medical-math_llama3_lora_adapter")

!zip -r medical-math_llama3_lora_adapter.zip ./medical-math_llama3_lora_adapter

# folder path in google drive
folder_path = "/content/drive/MyDrive/Fine-Tuning-Notebooks/Low-Rank-Adaptation-(LoRA)/Llama-3.2-3B-Instruct/Medical-Math"

# create folder if it doesnt exist
os.makedirs(folder_path, exist_ok=True)

# move zip file into google drive folder
shutil.move("medical-math_llama3_lora_adapter.zip", f"{folder_path}/medical-math_llama3_lora_adapter.zip")

# confirm saved
print("Adapter zip saved to Google Drive")

  adding: medical-math_llama3_lora_adapter/ (stored 0%)
  adding: medical-math_llama3_lora_adapter/adapter_config.json (deflated 58%)
  adding: medical-math_llama3_lora_adapter/tokenizer_config.json (deflated 43%)
  adding: medical-math_llama3_lora_adapter/tokenizer.json (deflated 85%)
  adding: medical-math_llama3_lora_adapter/chat_template.jinja (deflated 71%)
  adding: medical-math_llama3_lora_adapter/adapter_model.safetensors (deflated 7%)
  adding: medical-math_llama3_lora_adapter/README.md (deflated 65%)
Adapter zip saved to Google Drive
